# 11 · Preprocessing for the 5-class soil model (USCS × SPT-N)

**Design report section 11.** This project follows the geology-guided framework of Khamjan et al. (2026, *Engineering Geology* 371:108880), where the soil class combines the USCS group with strength, but validates every model with **spatial block cross-validation**. It is independent of the binary Clay/Sand project: it reads `config/project_5class.toml` and writes only to `data/processed_5class/`, `data/folds_5class/` and `outputs_5class/`.

| Step | Rule |
|---|---|
| Source | the same borehole CSV, checked against its SHA-256 |
| Sample | one record with a USCS symbol of either group **and** an SPT-N value, starting above 20 m, no longer than 3 m. Its depth is the top of the sampled interval |
| Groups | Clay = CL, CH; Sand = SC, SM, SP (SM, CH and SP are too rare in Downtown to be classes of their own) |
| Fine classes | Clay by SPT-N: Soft ≤ 4, Medium stiff 5–8, Stiff 9–15, Very stiff to hard > 15. Sand: Loose ≤ 10, Medium dense 11–30, Dense > 30 (Terzaghi & Peck ranges) |
| Merge rule | fixed before modelling, on counts only: while the smallest class of a group has fewer than 100 development samples, merge it with its smaller neighbour |
| Predictors | `UTM_Easting_m`, `UTM_Northing_m`, `Top_Elevation_m = ground elevation − sample top depth` |
| Gap filling | none (SPT-N is a point test; it cannot be carried between tests) |
| External holdout | the same 11 Downtown boreholes as the binary project (same rule, same lock) |
| Training extent | Downtown development boreholes only (scenario A) |

In [ ]:
import os, sys, json
from pathlib import Path

# 5-class soil model: this line selects config/project_5class.toml (keep it).
os.environ["SIEMREAP_CONFIG"] = "project_5class.toml"
# Run mode: "full" for the experiment, "smoke" for the reduced software validation run.
# os.environ["SIEMREAP_MODE"] = "smoke"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "project.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

from siemreap import config
cfg = config.load_config(ROOT)
MODE = config.run_mode()
print(config.smoke_banner(), "|", cfg["_config_name"])

from siemreap import data, geo, plots
AGG = config.stage_dir(cfg, "01_preprocessing", "aggregate")
RES = config.stage_dir(cfg, "01_preprocessing", "restricted")
PROC = config.processed_dir(cfg)

## 1. Raw data and samples

In [ ]:
raw = data.load_raw(cfg)          # stops if the file differs from the configured checksum
bh = data.borehole_table(raw, cfg)
samples, info = data.samples(raw, cfg)
print(json.dumps(info, indent=1))
display(samples.groupby("Region").agg(boreholes=("BH_Code", "nunique"), samples=("y_bin", "size"),
                                      clay_share=("y_bin", "mean"), spt_n_median=("SPT_N", "median")).round(3))

## 2. External holdout (same boreholes as the binary project, locked)

In [ ]:
flags = data.manual_flags(cfg)
external, sel_info = data.select_external(bh, cfg, flags)
lock = data.lock_external(cfg, external, sel_info)     # stops if the selection differs from the committed lock
print("locked SHA-256:", lock["sha256"])
display(external.groupby("group").agg(boreholes=("site", "size"), min_isolation_m=("site_isolation_m", "min")))
data.write_csv(external, PROC / "external_holdout.csv")

## 3. Classes: fine counts, merge rule, final classes

The counts below use the **development** boreholes only (the external boreholes' labels are not looked at). The merge rule is applied to these counts; no model or score is involved.

In [ ]:
mt = data.model_table(samples, external.index, cfg, flags)
dev_A = mt[mt.in_A]
fine = data.fine_classes(cfg)
counts = dev_A.fine.value_counts().reindex(range(len(fine)), fill_value=0)
display(pd.DataFrame({"group": [f["group"] for f in fine], "fine class": [f["name"] for f in fine],
                      "SPT-N": [f"<= {int(f['hi'])}" if not np.isfinite(f['lo']) else (f">= {int(f['lo']) + 1}" if not np.isfinite(f['hi'])
                                 else f"{int(f['lo']) + 1}-{int(f['hi'])}") for f in fine],
                      "development samples": counts.values,
                      "boreholes": [dev_A[dev_A.fine == i].BH_Code.nunique() for i in range(len(fine))]}))
merge = data.merge_classes(counts, cfg)
(PROC / "classes.json").write_text(json.dumps(merge, indent=1))
mt = data.apply_classes(mt, merge)
final_tab = pd.DataFrame(merge["classes"])
final_tab["boreholes"] = [mt[mt.in_A & (mt.y == i)].BH_Code.nunique() for i in range(len(final_tab))]
display(final_tab)
assert (final_tab.records >= cfg["target"]["min_class_records"]).all() or len(final_tab) <= 2

In [ ]:
CLS = merge["classes"]; NAMES = [c["name"] for c in CLS]; COLOURS = plots.class_colours(CLS)
fig, ax = plt.subplots(figsize=(5.4, 5.6), dpi=150)
plots.class_share_by_depth(mt[mt.in_A], NAMES, COLOURS, cfg["spatial"]["slice_m"], cfg["depth"]["bottom_m"], ax=ax)
ax.set_title("Development samples: class share by 1.5 m slice", loc="left", fontsize=10)
fig.savefig(AGG / "fig_class_share_by_depth.png", bbox_inches="tight")
display(pd.crosstab(pd.cut(mt[mt.in_A].depth_top_m, cfg["depth"]["bands_m"], right=False), mt[mt.in_A].class_name))

## 4. Prediction domain (geometry only)

In [ ]:
poly = geo.load_polygon(cfg, "downtown_shp")
gx, gy = geo.grid_columns(poly, cfg["grid3d"]["spacing_m"])
pred_cols = pd.DataFrame({"X": gx, "Y": gy})
data.write_csv(pred_cols, PROC / "prediction_columns.csv", index=False)
bhs = data.boreholes_of(mt)
d = geo.nearest_distance(pred_cols.values, bhs[bhs.in_A][["UTM_Easting_m", "UTM_Northing_m"]].values)
pdist = {"columns": int(len(d)), **{f"p{q}": round(float(np.percentile(d, q)), 1) for q in (5, 25, 50, 75, 95)},
         **{f"share_within_{t}m": round(float((d <= t).mean()), 4) for t in (100, 200, 300, 500, 1000)}}
print(json.dumps(pdist, indent=1))

## 5. Write processed files and the R settings

In [ ]:
data.write_csv(mt, PROC / "cells_all.csv", index=False)
dev = mt[mt.set == "development"]
r_cols = ["BH_Code", "UTM_Easting_m", "UTM_Northing_m", "depth_top_m", "Top_Elevation_m", "y", "y_bin", "class_name", "Region"]
data.write_csv(dev[r_cols].assign(is_downtown=dev.is_downtown.astype(int)), PROC / "cells_development.csv", index=False)
data.write_csv(bhs, PROC / "boreholes.csv")
proc_rel = cfg["paths"]["processed"]
r_settings = {"mode": MODE, "crs_epsg": cfg["data"]["crs_epsg"], "seed": cfg["seeds"]["global"],
              "site_linkage_m": cfg["external"]["site_linkage_m"], "blockcv": cfg["blockcv"], "spatial": cfg["spatial"],
              "outputs_dir": cfg["paths"]["outputs"],
              "classes": [{"name": c["name"], "group": c["group"]} for c in CLS],
              "files": {"cells": f"{proc_rel}/cells_development.csv", "pred_points": f"{proc_rel}/prediction_columns.csv",
                        "folds_dir": cfg["paths"]["folds"]}}
(PROC / "r_settings.json").write_text(json.dumps(r_settings, indent=1))
summary = {"samples": info, "external_selection": sel_info, "external_sha256": lock["sha256"],
           "fine_counts_development": counts.to_dict(), "classes": CLS,
           "samples_by_set": mt.set.value_counts().to_dict(),
           "scenario_A": {"boreholes": int(bhs.in_A.sum()), "samples": int(mt.in_A.sum())},
           "prediction_distance_to_development_boreholes_m": pdist, "environment": config.environment_record()}
(AGG / "preprocessing_summary.json").write_text(json.dumps(summary, indent=1, default=str))
print("written:", sorted(p.name for p in PROC.iterdir()))

## Next step: R (here, or in RStudio)

R decides the block size and builds the folds, with the original blockCV package:

1. `R/01_mc_block_size.R` measures the horizontal autocorrelation range of every class indicator in every 1.5 m depth slice with `blockCV::cv_spatial_autocor` (indicators present in fewer than 10 boreholes of a slice are skipped, as their variogram has no defined range). **The block size is the median of these ranges**, the way blockCV combines several layers (Valavi et al. 2019). Nobody chooses it by hand, and no model is involved.
2. `R/02_blockcv_folds.R` builds k = 5 folds with square blocks of that size, with random block-to-fold assignment balanced on the class (300 iterations, fixed seed).

The cell below runs them if it finds `Rscript`. Otherwise open `SiemReapML.Rproj` in RStudio and run:

```
Sys.setenv(SIEMREAP_PROCESSED = "data/processed_5class")
source("R/00_install_packages.R")   # once
source("R/01_mc_block_size.R")
source("R/02_blockcv_folds.R")
```

In [ ]:
import glob, shutil, subprocess

RUN_R_FROM_NOTEBOOK = True
RSCRIPT = None   # set the full path if R is not found, e.g. r"C:\Program Files\R\R-4.4.1\bin\Rscript.exe"

def find_rscript():
    if RSCRIPT:
        return RSCRIPT
    found = shutil.which("Rscript")
    if found:
        return found
    candidates = sorted(glob.glob(r"C:\Program Files\R\R-*\bin\Rscript.exe"))
    return candidates[-1] if candidates else None

rscript = find_rscript()
env = {**os.environ, "SIEMREAP_PROCESSED": cfg["paths"]["processed"]}
if not RUN_R_FROM_NOTEBOOK:
    print("Skipped: run the R scripts in RStudio (see above).")
elif rscript is None:
    print("Rscript not found. Set RSCRIPT above, or run the scripts in RStudio (see above).")
else:
    print("Rscript:", rscript)
    for script in ["R/00_install_packages.R", "R/01_mc_block_size.R", "R/02_blockcv_folds.R"]:
        print("\n>>", script, flush=True)
        with subprocess.Popen([rscript, script], cwd=ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding="utf-8", errors="replace") as proc:
            for line in proc.stdout:
                print(line, end="", flush=True)
        if proc.returncode != 0:
            raise RuntimeError(f"{script} failed (see the messages above)")
    print("\nR steps finished: continue with 12_folds_review_5class.ipynb")